# 21 · Group points in H3 cells against h3-py

**Question.** Does Fieldwork's `hex_aggregate` widget index points to the right cells, and what
does its sparse-cell rule do to the data?

**This check is weaker than most here, and says so.** Fieldwork indexes with `h3-js` and the
reference with `h3-py` — two bindings of the same C library. The indexing is therefore *not*
independently verified. What is verified is that Fieldwork calls it correctly, orders the result
as it claims, and applies its own sparse-cell rule, which is the part that is Fieldwork's alone.

**Two resolutions, because the example's own setting does not exercise the rule it depends on.**
At resolution 9 all 250 deaths fall into five cells, every one above the minimum, so nothing is
dropped. Resolution 11 produces single-point cells — the case that makes points vanish.

Criteria fixed before running: per resolution, the same cells kept, the same count in each, each
boundary and centre within 1e-9 degrees of h3-py's, the cells ordered by H3 index, and the
matched and dropped point counts exact.

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["h3"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

## Parity

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_21_hex import run

report = run(Path.cwd() / "fixtures" / "hex-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"], "over", report["points"], "points, minimum occupancy",
      report["minOccupancy"])
print()
print(f"{'resolution':>11} {'cells kept':>11} {'cells dropped':>14} {'points removed':>15} {'worst coord diff':>18}")
for case in report["cases"]:
    worst = max(case["worstBoundaryDifferenceDegrees"], case["worstCentreDifferenceDegrees"])
    print(f"{case['resolution']:>11} {case['cellsKeptReference']:>11} "
          f"{case['cellsDroppedBySparseRule']:>14} {case['pointsDroppedBySparseRule']:>15} "
          f"{worst:>18.1e}")
print()
print(report["environment"]["note"])

## Dive deeper · choosing the resolution, and for what

The widget takes `resolution` and `minOccupancy` as two independent numbers with no stated
relationship to the study area or to **why** the points are being aggregated. They are not
independent, and the purpose decides them.

Two purposes pull in **opposite** directions.

- **Geoprivacy.** Coarser cells hold more points, so no cell points at an individual. You want
  high occupancy and, crucially, **nothing dropped** — because the points the sparse rule
  removes are the most isolated ones, which are precisely the people most identifiable.
- **Summarization.** Finer cells show more pattern. You want enough cells across the study area
  to see structure, and few enough sparse ones that the picture is not mostly holes.

Measured on this study area — the Soho reporting rectangle, 1,318 m across its diagonal — with
250 deaths and a minimum occupancy of 2.

In [ ]:
import statistics
import h3

fixture = json.loads((Path.cwd() / "fixtures" / "hex-soho-001.json").read_text())
points = [p["coordinates"] for p in fixture["input"]["points"] if p["coordinates"]]
minimum = fixture["declaredParameters"]["minOccupancy"]
across_m = 1318   # the reporting rectangle's diagonal, from the clip fixture's boundary

print(f"{'res':>4} {'edge m':>8} {'cells':>6} {'kept':>5} {'deaths withheld':>16} "
      f"{'median occ':>11} {'cells across':>13}")
for resolution in range(7, 13):
    counts = {}
    for longitude, latitude in points:
        cell = h3.latlng_to_cell(latitude, longitude, resolution)
        counts[cell] = counts.get(cell, 0) + 1
    kept = {c: n for c, n in counts.items() if n >= minimum}
    withheld = sum(n for n in counts.values() if n < minimum)
    edge = h3.average_hexagon_edge_length(resolution, unit="m")
    print(f"{resolution:>4} {edge:>8.1f} {len(counts):>6} {len(kept):>5} "
          f"{withheld:>7} ({withheld / len(points):>5.1%}) {statistics.median(counts.values()):>11.1f} "
          f"{across_m / (edge * 2):>13.0f}")

### Reading that table for each purpose

**For geoprivacy, resolution 9 and coarser.** Nothing is withheld, and the median cell holds 17
deaths. The geoprivacy example uses exactly this, which is the right choice for what it is.

**For summarization, resolution 10.** Nine cells span the study area — enough to show where the
deaths concentrate — and the rule withholds one death out of 250. Resolution 11 shows more
structure and withholds 13; resolution 12 withholds **94 of 250**, more than a third, and the
median cell holds one point so most cells vanish before they are drawn.

**Resolution 7 and 8 are not aggregation at all** on an area this size: one or two cells larger
than the study area, which answers no question that was asked.

So the rule of thumb the widget does not state: pick the resolution from the **study area's
extent**, aiming for roughly ten cells across for summarization and as few as three for
privacy, then check what the minimum occupancy withholds at that choice.

### The sparse rule means two different things

This is the part that is easy to miss, and it is why the purpose has to be declared rather than
inferred.

The points the rule removes are the **most isolated** ones. For **geoprivacy** that is arguably
the right behaviour: the individuals easiest to identify are the ones withheld. For
**summarization** it is the opposite of what you want — the outliers are often the signal, and a
cholera death on its own two streets from the cluster is exactly the observation that matters.

The widget applies one rule either way, and nothing in a workflow records which purpose was
intended. Measured here: at resolution 11 the output represents 237 of 250 deaths, and a reader
summing the cells gets 237 with nothing on the map to say so.

### And what it does not buy

Occupancy is **not** an anonymity guarantee, and the widget's own note says so. A minimum of two
means no cell holds exactly one point. It does not mean a person cannot be identified from a
cell plus anything else a reader already knows — a street, a date, a household. That is a claim
about an adversary's knowledge, and no check and no threshold can settle it.

### Reading the result

`agrees` means Fieldwork and h3-py put the same points in the same cells with the same
boundaries, and that Fieldwork's own sparse rule was applied as declared. It does not mean H3
indexing is correct — both sides call the same C library — and it does not mean the aggregation
protects anybody.